# **FINAL PROJECT**

### Workaround codes for Web scraping and functions development 

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
import time
import random
from datetime import datetime

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# ─────────────────────────────────────────────
# Step 1 - Collect campaign URLs from listing pages
# ─────────────────────────────────────────────

def collect_urls(start_page=2, end_page=10):
    urls = set()
    
    for page_num in range(start_page, end_page + 1):
        print(f"Collecting page {page_num}...")
        response = requests.get(
            f"https://www.adsoftheworld.com/highlighted?page={page_num}",
            headers=headers
        )
        soup = BeautifulSoup(response.text, "html.parser")
        anchors = soup.find_all("a", href=lambda h: h and "/campaigns/" in h and not h.endswith("/new"))
        
        page_urls = set([a["href"] for a in anchors])
        urls.update(page_urls)
        
        print(f"  Page {page_num}: +{len(page_urls)} urls (total {len(urls)})")
        time.sleep(random.uniform(1, 2))  # polite delay between pages
    
    # Convert relative to absolute URLs
    return [f"https://www.adsoftheworld.com{u}" if u.startswith("/") else u for u in urls]


# ─────────────────────────────────────────────
# Step 2 - Scrape a single campaign page
# ─────────────────────────────────────────────

def scrape_campaign(url):
    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")

        title   = soup.select_one("h1").get_text(strip=True) if soup.select_one("h1") else ""
        brand   = soup.find("a", href=re.compile(r"/brands/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/brands/")) else ""
        agency  = soup.find("a", href=re.compile(r"/agencies/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/agencies/")) else ""
        country = soup.find("a", href=re.compile(r"/countries/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/countries/")) else ""
        #region  = soup.find("a", href=re.compile(r"/regions/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/regions/")) else ""
        medium  = soup.find("a", href=re.compile(r"/medium_types/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/medium_types/")) else ""
        industry = soup.find("a", href=re.compile(r"/industries/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/industries/")) else ""
        tags    = [a.get_text(strip=True) for a in soup.find_all("a", href=re.compile(r"/collections/"))]

        # Description - grab paragraphs longer than 40 chars
        desc_paragraphs = []
        for p in soup.find_all("p"):
            t = p.get_text(strip=True)
            if t and len(t) > 40:
                desc_paragraphs.append(t)
        description = "\n\n".join(desc_paragraphs[:5])

        # Thumbnail from og:image meta tag
        og_img = soup.find("meta", property="og:image")
        thumbnail_url = og_img["content"] if og_img and og_img.get("content") else ""

        # Published date and media count from AOTW summary sentence
        full_text = soup.get_text()
        date_match = re.search(r"published in .+? in ([A-Za-z]+,\s*\d{4})", full_text)
        media_match = re.search(r"contains (\d+) media asset", full_text)

        return {
            "id":             url.rstrip("/").split("/")[-1],
            "url":            url,
            "title":          title,
            "brand":          brand,
            "agency":         agency,
            "country":        country,
            #"region":         region,
            "medium":         medium,
            "industry":       industry,
            "description":    description,
            "tags":           tags,
            "thumbnail_url":  thumbnail_url,
            "published_date": date_match.group(1) if date_match else "",
            "media_count":    int(media_match.group(1)) if media_match else 0,
            "crawled_at":     datetime.utcnow().isoformat(),
        }

    except Exception as e:
        print(f"  Failed: {url} — {e}")
        return None


# ─────────────────────────────────────────────
# Step 3 - Run full pipeline
# ─────────────────────────────────────────────

def run(start_page=2, end_page=10, output="campaigns.json"):
    # Collect all URLs first
    all_urls = collect_urls(start_page=start_page, end_page=end_page)
    print(f"\nTotal unique URLs collected: {len(all_urls)}")

    # Scrape each campaign
    campaigns = []
    for i, url in enumerate(all_urls, 1):
        print(f"[{i}/{len(all_urls)}] Scraping: {url}")
        campaign = scrape_campaign(url)
        if campaign:
            campaigns.append(campaign)
            print(f"  ✓ {campaign['brand']} — {campaign['title'][:60]}")
        
        time.sleep(random.uniform(1, 2))  # polite delay between campaigns

        # Save checkpoint every 20 items
        if i % 20 == 0:
            with open(output, "w", encoding="utf-8") as f:
                json.dump(campaigns, f, ensure_ascii=False, indent=2)
            print(f"  [checkpoint saved — {len(campaigns)} campaigns]")

    # Final save
    with open(output, "w", encoding="utf-8") as f:
        json.dump(campaigns, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Done! {len(campaigns)} campaigns saved to {output}")
    return campaigns


# ─────────────────────────────────────────────
# Run it
# ─────────────────────────────────────────────

campaigns = run(start_page=4, end_page=10, output="campaigns.json")

In [ ]:
#Open file that was saved before
import json

with open("campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Replace id with numeric index
for i, campaign in enumerate(campaigns, 1): 
    campaign["id"] = i

with open("campaigns.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print(f"Updated {len(campaigns)} campaigns with numeric ids")
print(campaigns[0]) 

In [ ]:
import json
with open("campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

print(f"Loaded {len(campaigns)} campaigns")
print(campaigns[0]) 

In [ ]:
from rank_bm25 import BM25Okapi

# Combine fields into one string per campaign for BM25
def build_corpus(campaigns):
    corpus = []
    for c in campaigns:
        text = f"{c['title']} {c['brand']} {c['agency']} {c['country']} {c['industry']} {c['medium']} {c['description']}"
        tokens = text.lower().split()  # BM25 works on token list
        corpus.append(tokens)
    return corpus

corpus = build_corpus(campaigns)
bm25 = BM25Okapi(corpus)

print(f"BM25 index built with {len(corpus)} documents")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pre-trained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Combine fields into one string per campaign for embedding
def build_semantic_corpus(campaigns):
    texts = []
    for c in campaigns:
        text = f"{c['title']}. {c['brand']}. {c['industry']}. {c['description']}"
        texts.append(text)
    return texts

texts = build_semantic_corpus(campaigns)

# Embed all campaigns — this may take a minute
print("Embedding campaigns...")
embeddings = model.encode(texts, show_progress_bar=True)

# Save embeddings to disk so you don't have to re-run every time
#np.save("embeddings.npy", embeddings)

print(f"Embeddings shape: {embeddings.shape}")  # should be (num_campaigns, 384)

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# Load campaigns
with open("campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Load embeddings
embeddings = np.load("embeddings.npy")

# Rebuild BM25 index (no file to save/load, rebuilds instantly from campaigns)
def build_corpus(campaigns):
    corpus = []
    for c in campaigns:
        text = f"{c['title']} {c['brand']} {c['agency']} {c['country']} {c['industry']} {c['medium']} {c['description']}"
        tokens = text.lower().split()
        corpus.append(tokens)
    return corpus

corpus = build_corpus(campaigns)
bm25 = BM25Okapi(corpus)

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Campaigns: {len(campaigns)}")
print(f"Embeddings shape: {embeddings.shape}")
print("BM25 index ready")
print("Model loaded")

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Load embeddings if not already in memory
embeddings = np.load("embeddings.npy")

def hybrid_search(query, campaigns, bm25, embeddings, model, top_k=5, bm25_weight=0.5, semantic_weight=0.5):
    # BM25 scores
    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)

    # Semantic scores
    query_embedding = model.encode([query])
    semantic_scores = np.dot(embeddings, query_embedding.T).flatten()  # cosine-like similarity

    # Normalize both scores to 0-1 range so they are comparable
    bm25_scores_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)
    semantic_scores_norm = (semantic_scores - semantic_scores.min()) / (semantic_scores.max() - semantic_scores.min() + 1e-9)

    # Combine
    final_scores = bm25_weight * bm25_scores_norm + semantic_weight * semantic_scores_norm

    # Get top K indices
    top_indices = np.argsort(final_scores)[::-1][:top_k]

    # Return results
    results = []
    for idx in top_indices:
        results.append({
            "score": round(float(final_scores[idx]), 4),
            "id": campaigns[idx]["id"],
            "title": campaigns[idx]["title"],
            "brand": campaigns[idx]["brand"],
            "industry": campaigns[idx]["industry"],
            "description": campaigns[idx]["description"], 
            "url": campaigns[idx]["url"],
        })
    return results


# Test it
results = hybrid_search("Adidas", campaigns, bm25, embeddings, model, top_k=5)

for r in results:
    print(f"[{r['score']}] {r['brand']} — {r['title']}")
    print(f"  {r['description']}\n")

In [ ]:

import os
from google import genai

google_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

conversation_history = []

def format_campaigns_as_context(results):
    context = ""
    for i, r in enumerate(results, 1):
        context += f"""
Campaign {i}:
- Title: {r['title']}
- Brand: {r['brand']}
- Industry: {r['industry']}
- Score: {r['score']}
- Description: {r['description']}
- URL: {r['url']}
---
"""
    return context

def chat(user_message):
    results = hybrid_search(user_message, campaigns, bm25, embeddings, model, top_k=10)
    context = format_campaigns_as_context(results)

    system_prompt = f"""You are a marketing campaign expert assistant.
You help users find and analyze advertising campaigns from 'campaigns.json'.
Answer based on the retrieved campaigns below. 
Answer questions by providing the campaign name, brand, industry, description, url to the campaign.
If the user asks something unrelated to the campaigns, politely redirect them.

Retrieved campaigns:
{context}"""

    conversation_history.append({
        "role": "user",
        "parts": [{"text": user_message}]
    })

    response = google_client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=conversation_history,
        config={
            "system_instruction": system_prompt,
            "max_output_tokens": 1000,
        }
    )

    assistant_message = response.text

    
    conversation_history.append({
        "role": "model",
        "parts": [{"text": assistant_message}]
    })

    return assistant_message


# Chat loop
print("Marketing Campaign Assistant (type 'quit' to exit)\n")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", ""]:
        break
    response = chat(user_input)
    print(f"\nAssistant: {response}\n")

In [ ]:
import streamlit as st
import os
from google import genai

# --- Keep all functions ---
# (copy: load campaigns, build bm25, embeddings, hybrid_search, format_campaigns_as_context)

# --- Streamlit UI ---
st.title("Marketing Campaign Assistant")


google_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# Save conversation history in session
if "conversation_history" not in st.session_state:
    st.session_state.conversation_history = []

if "messages" not in st.session_state:
    st.session_state.messages = []

# Show chat history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Input from user
user_input = st.chat_input("Ask about campaigns...")

if user_input:
    # Show user message
    with st.chat_message("user"):
        st.write(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})

    # Call chat function
    results = hybrid_search(user_input, campaigns, bm25, embeddings, model, top_k=10)
    context = format_campaigns_as_context(results)

    system_prompt = f"""You are a marketing campaign expert assistant...
Retrieved campaigns:
{context}"""

    st.session_state.conversation_history.append({
        "role": "user",
        "parts": [{"text": user_input}]
    })

    response = google_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=st.session_state.conversation_history,
        config={
            "system_instruction": system_prompt,
            "max_output_tokens": 1000,
        }
    )

    assistant_message = response.text
    st.session_state.conversation_history.append({
        "role": "model",
        "parts": [{"text": assistant_message}]
    })

    
    with st.chat_message("assistant"):
        st.write(assistant_message)
    st.session_state.messages.append({"role": "assistant", "content": assistant_message})


In [ ]:
#check model
models = google_client.models.list()
for m in models:
    print(f"Model Name: {m.name} - Supported Methods: {m.supported_methods}")
    # show on Streamlit
    st.write(f"Model: {m.name}")

In [ ]:
import json
import os
import re

with open("project/campaigns.json", "r", encoding="utf-8") as f:
      campaigns = json.load(f)

print(f"Loaded {len(campaigns)} campaigns")

In [ ]:
patterns = {
    "AOTW auto-sentence": r"This\s+professional campaign titled",
    "Newsletter CTA":     r"Don't miss out",
    "Credit line":        r"Client\s*:",
}

for pattern_name, pattern in patterns.items():
    matches = [c for c in campaigns if re.search(pattern, c.get("description", ""))]
    print(f"{pattern_name}: found in {len(matches)}/{len(campaigns)} campaigns")
    if matches:
        sample = matches[0]
        print(f"  Example: [{sample['id']}] {sample['title'][:50]}")
        snippet = re.search(pattern, sample['description']).group(0)
        print(f"  Matched: '{snippet}'\n")

In [ ]:
#find campaigns that desciption only contain junk info
import re

def has_real_description(desc):
    """
    temporarily junk patterns to see if there are any campaigns that have real description
    """
    text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", desc, flags=re.DOTALL)
    text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
    text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
    text = text.strip()
    return len(text) > 50  # has at least 50 chars

no_real_desc = [c for c in campaigns if not has_real_description(c.get("description", ""))]
has_real_desc = [c for c in campaigns if has_real_description(c.get("description", ""))]

print(f"has real desc: {len(has_real_desc)}/424")
print(f"only junk/empty: {len(no_real_desc)}/424")


print("\n--- Examples of campaigns without original description ---")
for c in no_real_desc[:3]:
    print(f"\n[{c['id']}] {c['title'][:50]}")
    print(f"original desc: {c['description'][:200]}")

print("\n--- Examples of campaigns with original description ---")
for c in has_real_desc[:3]:
    print(f"\n[{c['id']}] {c['title'][:50]}")
    print(f"original desc: {c['description'][:200]}")

In [ ]:
import re

# Take campaign [2] description
c2 = [c for c in campaigns if c["id"] == 2][0]
desc = c2["description"]

print("=== ORIGINAL ===")
print(repr(desc))

# Apply cleaning step by step
text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", desc, flags=re.DOTALL)
print("\n=== AFTER removing auto-sentence ===")
print(repr(text))

text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
print("\n=== AFTER removing newsletter CTA ===")
print(repr(text))

text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
print("\n=== AFTER removing credit line ===")
print(repr(text))

print(f"\n=== FINAL length: {len(text.strip())} chars ===")

In [ ]:
import re
import json

def clean_description(text: str) -> str:
    text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", text, flags=re.DOTALL)
    text = re.sub(r"It was submitted .{3,30} ago\.", "", text)
    text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
    text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Load original
with open("project/campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Clean descriptions in place
for c in campaigns:
    c["description"] = clean_description(c.get("description", ""))

# Save to new file — original untouched
with open("project/campaigns_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print(f"Done. {len(campaigns)} campaigns cleaned → campaigns_cleaned.json")

In [ ]:
c2 = [c for c in campaigns if c["id"] == 2][0]
print(len(c2["description"]))  # 51
print(51 < 70)                 # should print True

In [ ]:
short = [c for c in campaigns if len(c["description"]) < 70]
print(f"Count: {len(short)}")
for c in short:
    print(f"[{c['id']}] len={len(c['description'])} — {repr(c['description'])}")

In [ ]:
def clean_description(text: str) -> str:
    text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", text, flags=re.DOTALL)
    text = re.sub(r"It was submitted .+?ago[^.]*\.", "", text)  # fixed
    text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
    text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Re-apply to original campaigns
with open("project/campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

for c in campaigns:
    c["description"] = clean_description(c.get("description", ""))

with open("project/campaigns_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print("Done. Re-cleaned and saved.")

In [ ]:
c67 = [c for c in campaigns if c["id"] == 67][0]
print(repr(c67["description"]))  # should be empty or just the real content

In [ ]:
import json

with open("project/campaigns_cleaned.json", "r", encoding="utf-8") as f:
    old_campaigns = json.load(f)

new_campaigns = []
for i, c in enumerate(old_campaigns):
    new = {
        "metadata": {
            "id":             i + 1,
            "url":            c.get("url", ""),
            "title":          c.get("title", ""),
            "brand":          c.get("brand", ""),
            "agency":         c.get("agency", ""),
            "industry":       c.get("industry", ""),
            "country":        c.get("country", ""),
            "medium":         c.get("medium", ""),
            "tags":           c.get("tags", []),
            "published_date": c.get("published_date", ""),
        },
        "content": {
            "description":   c.get("description", ""),
            "thumbnail_url": c.get("thumbnail_url", ""),
            "media_count":   c.get("media_count", 0),
        },
        "ai_enrichment": {
            "concept_summary": "",
            "target_audience": "",
            "tactics":         []
        },
        "system": {
            "crawled_at":  c.get("crawled_at", ""),
            "is_enriched": False
        }
    }
    new_campaigns.append(new)

with open("project/campaigns_v2.json", "w", encoding="utf-8") as f:
    json.dump(new_campaigns, f, ensure_ascii=False, indent=2)

print(f"Done. {len(new_campaigns)} campaigns migrated → campaigns_v2.json")

In [ ]:
import json
import time
import os
import re
from google import genai
from google.genai import types

# Load file FIRST
with open("project/campaigns_v2.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

ENRICH_PROMPT = """
Based on given campaign information, analyze it and return ONLY a valid JSON object, no markdown, no backticks:

Campaign: "{title}" by {brand}, medium: {medium}, industry: {industry}
Description: {description}

{{
  "concept_summary": "1-2 sentences describing the core creative idea",
  "target_audience": "Who this campaign targets, be specific e.g. Thai street food lovers aged 18-35",
  "execution_tactics":"1-2 sentences, focus on channels, activations e.g. Hero film on YouTube + TV",
  "objective": "1 sentence describing objective of the campaign? e.g. Brand awareness, sales, engagement, etc."
}}
"""

def enrich_campaign(c):
    meta = c["metadata"]
    desc = c["content"]["description"]
    
    prompt = ENRICH_PROMPT.format(
        title=meta.get("title", ""),
        brand=meta.get("brand", ""),
        industry=meta.get("industry", ""),
        medium=meta.get("medium", ""),
        description=desc[:800]
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[{"role": "user", "parts": [{"text": prompt}]}],
        config=types.GenerateContentConfig(max_output_tokens=1500),
    )
    
    raw = response.text.strip()
    print(f"=== RAW RESPONSE ===")
    print(repr(raw))
    print(f"Length: {len(raw)}")
    print(f"Finish reason: {response.candidates[0].finish_reason}")
    print(f"====================")
    
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    
    return json.loads(raw.strip())

# Test on 2 campaigns only
total = len(campaigns)
enriched_count = 0

for i, c in enumerate(campaigns):
    if c["system"]["is_enriched"]:
        print(f"[{i+1}/{total}] Skipping: {c['metadata']['title'][:50]}")
        continue

    try:
        enrichment = enrich_campaign(c)
        c["ai_enrichment"] = enrichment
        c["system"]["is_enriched"] = True
        enriched_count += 1
        print(f"[{i+1}/{total}] ✓ {c['metadata']['title'][:50]}")
        #print(json.dumps(enrichment, indent=2, ensure_ascii=False))
    except Exception as e:
        print(f"[{i+1}/{total}] ✗ Failed: {c['metadata']['title'][:50]} — {e}")
    
    if (i + 1) % 20 == 0:
        with open("project/campaigns_v2.json", "w", encoding="utf-8") as f:
            json.dump(campaigns, f, ensure_ascii=False, indent=2)
        print(f"  [checkpoint: {enriched_count} enriched so far]")

    time.sleep(1)

with open("project/campaigns_v2.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print(f"\nDone. {enriched_count}/{total} campaigns enriched.")

In [ ]:
with open("project/campaigns_v2.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

enriched = [c for c in campaigns if c["system"]["is_enriched"]]
not_enriched = [c for c in campaigns if not c["system"]["is_enriched"]]

print(f"Enriched: {len(enriched)}/424")
print(f"Not enriched (failed or MAX_TOKENS): {not_enriched}")

In [ ]:
import json

with open("project/campaigns_v2.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Check 3 random campaigns
import random
samples = random.sample(campaigns, 3)
for c in samples:
    print(f"\n=== {c['metadata']['title'][:50]} ===")
    print(json.dumps(c["ai_enrichment"], indent=2, ensure_ascii=False))

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# Load new schema
with open("project/campaigns_v2.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

def _campaign_text(c: dict) -> str:
    meta = c.get("metadata", {})
    content = c.get("content", {})
    ai = c.get("ai_enrichment", {})
    
    tactics = ""
    if isinstance(ai.get("execution_tactics"), list):
        tactics = " ".join(ai.get("execution_tactics", []))
    elif isinstance(ai.get("execution_tactics"), str):
        tactics = ai.get("execution_tactics", "")

    return (
        f"{meta.get('title', '')} "
        f"{meta.get('brand', '')} "
        f"{meta.get('agency', '')} "
        f"{meta.get('country', '')} "
        f"{meta.get('industry', '')} "
        f"{meta.get('medium', '')} "
        f"{content.get('description', '')} "
        f"{ai.get('concept_summary', '')} "
        f"{ai.get('target_audience', '')} "
        f"{tactics} "
        f"{ai.get('objective', '')}"
    )

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [_campaign_text(c) for c in campaigns]

print(f"Building embeddings for {len(texts)} campaigns...")
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,
)

# Save to same folder as app.py
np.save("embeddings_v2.npy", embeddings)
print(f"Done. Shape: {embeddings.shape}")
print(f"Saved to embeddings_v2.npy")

Crawling from another source


In [ ]:
import json
import time
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime
 
# --- CONFIG ---
GALLERY_URL = "https://clios.com/winners-gallery/explore?vertical=Clio+Awards&season=2022"
BASE_URL = "https://clios.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
TEST_LIMIT = 80  
DELAY = 2  # Seconds between requests to be polite
 
 

def get_all_campaign_urls(base_gallery_url, max_pages=10,limit = None, start_page=1):
    """
    Crawl all pages of the gallery by incrementing page_number param.
    Stops when no more campaign URLs are found.
    """
    all_campaigns = []
    seen_urls = set()

    for page in range(start_page, start_page + max_pages):
        # Page 1 has no page_number param, page 2+ uses &page_number=N
        if page == 1:
            url = base_gallery_url
        else:
            url = f"{base_gallery_url}&page_number={page}"

        print(f"Fetching gallery page {page}: {url}")
        resp = requests.get(url, headers=HEADERS)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Extract campaign detail links
        found = 0
        for link in soup.find_all("a", href=re.compile(r"/winners-gallery/details/\d+")):
            href = link.get("href", "")
            full_url = href if href.startswith("http") else BASE_URL + href
            if full_url not in seen_urls:
                seen_urls.add(full_url)
                all_campaigns.append({"url": full_url})
                found += 1
         # Stop early if limit reached
            if limit and len(all_campaigns) >= limit:
                print(f"  Reached limit ({limit}). Stopping.")
                return all_campaigns

        print(f"  Found {found} new campaign URLs (total: {len(all_campaigns)})")

        if found == 0:
            print("  No more campaigns. Stopping.")
            break

        time.sleep(DELAY)

    return all_campaigns

# --- STEP 1: Get campaign URLs from gallery page ---
 
def get_campaign_urls(gallery_url, limit=None):
    """
    Fetch gallery page and extract detail page URLs.
    Returns list of dicts with basic info from gallery cards.
    """
    resp = requests.get(gallery_url, headers=HEADERS)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
 
    campaigns = []
    # Each campaign card links to /winners-gallery/details/ID
    for link in soup.find_all("a", href=re.compile(r"/winners-gallery/details/\d+")):
        href = link.get("href", "")
        if href and href not in [c["url"] for c in campaigns]:
            full_url = href if href.startswith("http") else BASE_URL + href
            campaigns.append({"url": full_url})
 
    if limit:
        campaigns = campaigns[:limit]
 
    print(f"Found {len(campaigns)} campaign URLs")
    return campaigns
 
 
# --- STEP 2: Parse a single campaign detail page ---
 
def parse_campaign_detail(url):
    """
    Fetch and parse a Clio campaign detail page.
    Returns a dict matching campaigns_v2.json schema.
    """
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
 
    # --- Extract metadata from the detail page ---
    # Title: usually in the page title "Brand: Title - The Clios"
    page_title = soup.find("title")
    title_text = page_title.text.strip() if page_title else ""
    # Parse "Brand: Title - The Clios" format
    title_parts = title_text.replace(" - The Clios", "").split(": ", 1)
    brand = title_parts[0].strip() if len(title_parts) > 1 else ""
    title = title_parts[1].strip() if len(title_parts) > 1 else title_parts[0].strip()
 
    # Find the detail section with Entry Type, Medium, Category, etc.
    # These are in the winner-slideshow-details div
    details_div = soup.find("div", id="winner-slideshow-details")
    
    metadata = {}
    if details_div:
        # Look for label-value pairs
        labels = details_div.find_all(class_=re.compile(r"font-medium|font-bold|text-ivory"))
        # Alternative: scan all text content
        detail_text = details_div.get_text(separator="\n").strip()
        lines = [l.strip() for l in detail_text.split("\n") if l.strip()]
        
        # Parse key-value pairs from lines
        key_fields = ["Entry Type", "Program", "Medium Type", "Medium", "Category",
                       "Entrant Company"]
        current_key = None
        for line in lines:
            if line in key_fields:
                current_key = line
            elif current_key:
                metadata[current_key] = line
                current_key = None
 
    # Country and year - look in the main content area
    # They appear as standalone text near the title
    country = ""
    year = "2022"
    
    # Search for country/year pattern in the slideshow area
    slideshow_div = soup.find("div", class_=re.compile(r"winner-slideshow"))
    if slideshow_div:
        all_text = slideshow_div.get_text(separator="\n")
        lines = [l.strip() for l in all_text.split("\n") if l.strip()]
        for line in lines:
            # Year is a 4-digit number
            if re.match(r"^20\d{2}$", line):
                year = line
            # Country detection: lines that are just a country name (no special chars, title case)
            elif re.match(r"^[A-Z][a-z]+(\s[A-Z][a-z]+)*$", line) and len(line) < 30:
                if line not in ["Grand", "Gold", "Silver", "Bronze", "Shortlist", "Winner",
                                "Share", "Credits", "Award"]:
                    country = line
 
    # Description - in winner-slideshow-description div
    desc_div = soup.find("div", id="winner-slideshow-description")
    description = ""
    if desc_div:
        description = desc_div.get_text(separator="\n").strip()
 
    # Thumbnail - look for campaign images
    thumbnail_url = ""
    img_tags = soup.find_all("img", src=re.compile(r"resized-media\.entries\.clios\.com"))
    if img_tags:
        thumbnail_url = img_tags[0].get("src", "")
 
    # Agency from credits section
    agency = metadata.get("Entrant Company", "")
 
    # Map medium from Clios to simpler categories
    medium_raw = metadata.get("Medium", "")
 
    return {
        "title": title,
        "brand": brand,
        "agency": agency,
        "country": country,
        "year": year,
        "medium": medium_raw,
        "category": metadata.get("Category", ""),
        "entry_type": metadata.get("Entry Type", ""),
        "program": metadata.get("Program", ""),
        "description": description,
        "thumbnail_url": thumbnail_url,
        "url": url,
    }
 
 
# --- STEP 3: Map to campaigns_v2.json schema ---
 
def map_to_schema(parsed, campaign_id):
    """
    Convert parsed Clios data to campaigns_v2.json schema.
    ai_enrichment will be filled later via Gemini.
    """
    return {
        "metadata": {
            "id": campaign_id,
            "url": parsed["url"],
            "title": parsed["title"],
            "brand": parsed["brand"],
            "agency": parsed["agency"],
            "industry": parsed["entry_type"],  # Map Entry Type → industry
            "country": parsed["country"],
            "medium": parsed["medium"],
            "tags": [parsed.get("program", ""), f"Clio {parsed.get('category', '')}"],
            "published_date": f"{parsed['year']}",
        },
        "content": {
            "description": parsed["description"],
            "thumbnail_url": parsed["thumbnail_url"],
            "media_count": 1,
        },
        "ai_enrichment": {
            "concept_summary": "",
            "target_audience": "",
            "execution_tactics": "",
            "objective": "",
        },
        "system": {
            "crawled_at": datetime.utcnow().isoformat(),
            "is_enriched": False,
            "source": "clios.com",
        },
    }
 
 
# --- STEP 4: Dedup check ---
 
def load_existing_titles(filepath):
    """Load existing campaign titles for deduplication."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
            return set(c["metadata"]["title"].lower().strip() for c in data)
    except FileNotFoundError:
        return set()
 
file_path = ['campaigns_v2.json','campaigns_new.json'] 
# --- MAIN: Run the crawl ---
 
def crawl_clios(limit=TEST_LIMIT):
    """
    Main crawl function.
    1. Get campaign URLs from gallery
    2. Fetch each detail page
    3. Map to schema
    4. Dedup against existing data
    5. Save to campaigns_new.json
    """
    existing_titles = set()
    # Load existing titles for dedup
    for filepath in file_path:
        existing_titles.update(load_existing_titles(filepath))
        print(f"Loaded {len(existing_titles)} existing campaign titles for dedup check")
 
    # Get campaign URLs
    campaign_urls = get_all_campaign_urls(GALLERY_URL,start_page=1)
    if limit:
        campaign_urls = campaign_urls[:limit]
 
    # Get next available ID (start after existing campaigns)
    try:
        with open("campaigns_v2.json", "r") as f:
            existing = json.load(f)
        next_id = max(c["metadata"]["id"] for c in existing) + 1
    except FileNotFoundError:
        next_id = 1
 
    new_campaigns = []
    skipped = 0
 
    for i, camp in enumerate(campaign_urls):
        print(f"Crawling {i+1}/{len(campaign_urls)}: {camp['url']}")
 
        try:
            parsed = parse_campaign_detail(camp["url"])
 
            # Dedup check
            if parsed["title"].lower().strip() in existing_titles:
                print(f"  SKIP (duplicate): {parsed['title']}")
                skipped += 1
                continue
 
            # Map to schema
            mapped = map_to_schema(parsed, next_id)
            new_campaigns.append(mapped)
            next_id += 1
 
            print(f"  OK: {parsed['title']} ({parsed['brand']}, {parsed['country']})")
 
        except Exception as e:
            print(f"  ERROR: {e}")
 
        # Be polite
        time.sleep(DELAY)
 
    # Save results
    output_file = "campaigns_new.json"
    existing_new = []
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            existing_new = json.load(f)
    except FileNotFoundError:
        existing_new = []

    existing_new.extend(new_campaigns)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(existing_new, f, indent=2, ensure_ascii=False)

    print(f"Appended {len(new_campaigns)} new campaigns. Total in file: {len(existing_new)}")
 
 
# Run it
if __name__ == "__main__":
    results = crawl_clios()
    # Print first result to verify
    if results:
        print("\n--- Sample output ---")
        print(json.dumps(results[0], indent=2, ensure_ascii=False))
 
 

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# Load campaigns
with open("campaigns_new.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Same function as app.py — must match exactly
def _campaign_text(c):
    meta = c.get("metadata", {})
    content = c.get("content", {})
    ai = c.get("ai_enrichment", {})

    tactics = ai.get("execution_tactics", "")
    if isinstance(tactics, list):
        tactics = " ".join(tactics)

    return (
        f"{meta.get('title', '')} "
        f"{meta.get('brand', '')} "
        f"{meta.get('agency', '')} "
        f"{meta.get('country', '')} "
        f"{meta.get('industry', '')} "
        f"{meta.get('medium', '')} "
        f"{content.get('description', '')} "
        f"{ai.get('concept_summary', '')} "
        f"{ai.get('target_audience', '')} "
        f"{tactics} "
        f"{ai.get('objective', '')}"
    )

# Build texts
texts = [_campaign_text(c) for c in campaigns]
print(f"Total campaigns: {len(texts)}")

# Embed using same model as app.py
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,
)

# Save
np.save("embeddings_new.npy", embeddings)
print(f"Done. Shape: {embeddings.shape}")
print(f"Saved to embeddings_new.npy")

In [ ]:
!pip install sentence_transformers

In [ ]:
import json
import time
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# --- CONFIG ---
GALLERY_URL = "https://clios.com/winners-gallery/explore?vertical=Clio+Awards&season=2022"
BASE_URL = "https://clios.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
TEST_LIMIT = 80
DELAY = 2  # Seconds between requests to be polite



def get_all_campaign_urls(base_gallery_url, max_pages=10,limit = None, start_page=1):
    """
    Crawl all pages of the gallery by incrementing page_number param.
    Stops when no more campaign URLs are found.
    """
    all_campaigns = []
    seen_urls = set()

    for page in range(start_page, start_page + max_pages):
        # Page 1 has no page_number param, page 2+ uses &page_number=N
        if page == 1:
            url = base_gallery_url
        else:
            url = f"{base_gallery_url}&page_number={page}"

        print(f"Fetching gallery page {page}: {url}")
        resp = requests.get(url, headers=HEADERS)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Extract campaign detail links
        found = 0
        for link in soup.find_all("a", href=re.compile(r"/winners-gallery/details/\d+")):
            href = link.get("href", "")
            full_url = href if href.startswith("http") else BASE_URL + href
            if full_url not in seen_urls:
                seen_urls.add(full_url)
                all_campaigns.append({"url": full_url})
                found += 1
         # Stop early if limit reached
            if limit and len(all_campaigns) >= limit:
                print(f"  Reached limit ({limit}). Stopping.")
                return all_campaigns

        print(f"  Found {found} new campaign URLs (total: {len(all_campaigns)})")

        if found == 0:
            print("  No more campaigns. Stopping.")
            break

        time.sleep(DELAY)

    return all_campaigns

# --- STEP 1: Get campaign URLs from gallery page ---

def get_campaign_urls(gallery_url, limit=None):
    """
    Fetch gallery page and extract detail page URLs.
    Returns list of dicts with basic info from gallery cards.
    """
    resp = requests.get(gallery_url, headers=HEADERS)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    campaigns = []
    # Each campaign card links to /winners-gallery/details/ID
    for link in soup.find_all("a", href=re.compile(r"/winners-gallery/details/\d+")):
        href = link.get("href", "")
        if href and href not in [c["url"] for c in campaigns]:
            full_url = href if href.startswith("http") else BASE_URL + href
            campaigns.append({"url": full_url})

    if limit:
        campaigns = campaigns[:limit]

    print(f"Found {len(campaigns)} campaign URLs")
    return campaigns


# --- STEP 2: Parse a single campaign detail page ---

def parse_campaign_detail(url):
    """
    Fetch and parse a Clio campaign detail page.
    Returns a dict matching campaigns_v2.json schema.
    """
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # --- Extract metadata from the detail page ---
    # Title: usually in the page title "Brand: Title - The Clios"
    page_title = soup.find("title")
    title_text = page_title.text.strip() if page_title else ""
    # Parse "Brand: Title - The Clios" format
    title_parts = title_text.replace(" - The Clios", "").split(": ", 1)
    brand = title_parts[0].strip() if len(title_parts) > 1 else ""
    title = title_parts[1].strip() if len(title_parts) > 1 else title_parts[0].strip()

    # Find the detail section with Entry Type, Medium, Category, etc.
    # These are in the winner-slideshow-details div
    details_div = soup.find("div", id="winner-slideshow-details")

    metadata = {}
    if details_div:
        # Look for label-value pairs
        labels = details_div.find_all(class_=re.compile(r"font-medium|font-bold|text-ivory"))
        # Alternative: scan all text content
        detail_text = details_div.get_text(separator="\n").strip()
        lines = [l.strip() for l in detail_text.split("\n") if l.strip()]

        # Parse key-value pairs from lines
        key_fields = ["Entry Type", "Program", "Medium Type", "Medium", "Category",
                       "Entrant Company"]
        current_key = None
        for line in lines:
            if line in key_fields:
                current_key = line
            elif current_key:
                metadata[current_key] = line
                current_key = None

    # Country and year - look in the main content area
    # They appear as standalone text near the title
    country = ""
    year = "2022"

    # Search for country/year pattern in the slideshow area
    slideshow_div = soup.find("div", class_=re.compile(r"winner-slideshow"))
    if slideshow_div:
        all_text = slideshow_div.get_text(separator="\n")
        lines = [l.strip() for l in all_text.split("\n") if l.strip()]
        for line in lines:
            # Year is a 4-digit number
            if re.match(r"^20\d{2}$", line):
                year = line
            # Country detection: lines that are just a country name (no special chars, title case)
            elif re.match(r"^[A-Z][a-z]+(\s[A-Z][a-z]+)*$", line) and len(line) < 30:
                if line not in ["Grand", "Gold", "Silver", "Bronze", "Shortlist", "Winner",
                                "Share", "Credits", "Award"]:
                    country = line

    # Description - in winner-slideshow-description div
    desc_div = soup.find("div", id="winner-slideshow-description")
    description = ""
    if desc_div:
        description = desc_div.get_text(separator="\n").strip()

    # Thumbnail - look for campaign images
    thumbnail_url = ""
    img_tags = soup.find_all("img", src=re.compile(r"resized-media\.entries\.clios\.com"))
    if img_tags:
        thumbnail_url = img_tags[0].get("src", "")

    # Agency from credits section
    agency = metadata.get("Entrant Company", "")

    # Map medium from Clios to simpler categories
    medium_raw = metadata.get("Medium", "")

    return {
        "title": title,
        "brand": brand,
        "agency": agency,
        "country": country,
        "year": year,
        "medium": medium_raw,
        "category": metadata.get("Category", ""),
        "entry_type": metadata.get("Entry Type", ""),
        "program": metadata.get("Program", ""),
        "description": description,
        "thumbnail_url": thumbnail_url,
        "url": url,
    }


# --- STEP 3: Map to campaigns_v2.json schema ---

def map_to_schema(parsed, campaign_id):
    """
    Convert parsed Clios data to campaigns_v2.json schema.
    ai_enrichment will be filled later via Gemini.
    """
    return {
        "metadata": {
            "id": campaign_id,
            "url": parsed["url"],
            "title": parsed["title"],
            "brand": parsed["brand"],
            "agency": parsed["agency"],
            "industry": parsed["entry_type"],  # Map Entry Type → industry
            "country": parsed["country"],
            "medium": parsed["medium"],
            "tags": [parsed.get("program", ""), f"Clio {parsed.get('category', '')}"],
            "published_date": f"{parsed['year']}",
        },
        "content": {
            "description": parsed["description"],
            "thumbnail_url": parsed["thumbnail_url"],
            "media_count": 1,
        },
        "ai_enrichment": {
            "concept_summary": "",
            "target_audience": "",
            "execution_tactics": "",
            "objective": "",
        },
        "system": {
            "crawled_at": datetime.utcnow().isoformat(),
            "is_enriched": False,
            "source": "clios.com",
        },
    }


# --- STEP 4: Dedup check ---

def load_existing_titles(filepath):
    """Load existing campaign titles for deduplication."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
            return set(c["metadata"]["title"].lower().strip() for c in data)
    except FileNotFoundError:
        return set()

file_path = ['campaigns_v2.json','campaigns_new.json']
# --- MAIN: Run the crawl ---

def crawl_clios(limit=TEST_LIMIT):
    """
    Main crawl function.
    1. Get campaign URLs from gallery
    2. Fetch each detail page
    3. Map to schema
    4. Dedup against existing data
    5. Save to campaigns_new.json
    """
    existing_titles = set()
    # Load existing titles for dedup
    for filepath in file_path:
        existing_titles.update(load_existing_titles(filepath))
        print(f"Loaded {len(existing_titles)} existing campaign titles for dedup check")

    # Get campaign URLs
    campaign_urls = get_all_campaign_urls(GALLERY_URL,start_page=1)
    if limit:
        campaign_urls = campaign_urls[:limit]

    # Get next available ID (start after existing campaigns)
    try:
        with open("campaigns_v2.json", "r") as f:
            existing = json.load(f)
        next_id = max(c["metadata"]["id"] for c in existing) + 1
    except FileNotFoundError:
        next_id = 1

    new_campaigns = []
    skipped = 0

    for i, camp in enumerate(campaign_urls):
        print(f"Crawling {i+1}/{len(campaign_urls)}: {camp['url']}")

        try:
            parsed = parse_campaign_detail(camp["url"])

            # Dedup check
            if parsed["title"].lower().strip() in existing_titles:
                print(f"  SKIP (duplicate): {parsed['title']}")
                skipped += 1
                continue

            # Map to schema
            mapped = map_to_schema(parsed, next_id)
            new_campaigns.append(mapped)
            next_id += 1

            print(f"  OK: {parsed['title']} ({parsed['brand']}, {parsed['country']})")

        except Exception as e:
            print(f"  ERROR: {e}")

        # Be polite
        time.sleep(DELAY)

    # Save results
    output_file = "campaigns_new.json"
    existing_new = []
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            existing_new = json.load(f)
    except FileNotFoundError:
        existing_new = []

    existing_new.extend(new_campaigns)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(existing_new, f, indent=2, ensure_ascii=False)

    print(f"Appended {len(new_campaigns)} new campaigns. Total in file: {len(existing_new)}")


# Run it
if __name__ == "__main__":
    results = crawl_clios()
    # Print first result to verify
    if results:
        print("\n--- Sample output ---")
        print(json.dumps(results[0], indent=2, ensure_ascii=False))
